# Amazon Bedrock 가드레일과 AgentCore 메모리를 활용한 대화 보안

## 개요

이 튜토리얼은 Amazon Bedrock 가드레일과 AgentCore 메모리를 통합하여 안전한 대화형 에이전트를 만드는 방법을 보여줍니다. 민감한 콘텐츠를 필터링하면서 상호작용 간 대화 컨텍스트를 유지하는 에이전트를 구축합니다.

### 튜토리얼 세부 정보

| 항목              | 세부 내용                                                         |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | 가드레일 / 메모리 통합                                            |
| 에이전트 유형       | 가드레일이 적용된 메모리 지원 에이전트                              |
| 에이전트 프레임워크  | Strands Agents                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                       |
| 주요 기능           | 가드레일, 메모리 통합, 콘텐츠 필터링                               |
| 예제 난이도         | 중급                                                              |
| 사용 SDK           | Amazon Bedrock Python SDK 및 Bedrock Memory SDK                   |

### 학습 내용

이 튜토리얼에서 학습할 내용:
1. 에이전트를 위한 메모리 리소스 생성 방법
2. 콘텐츠 필터링이 포함된 Amazon Bedrock 가드레일 구현 방법
3. 가드레일과 메모리 기능을 결합한 커스텀 훅 구축 방법
4. 안전한 대화 기록만 선택적으로 저장하는 방법
5. 다양한 유형의 콘텐츠로 보안 에이전트 테스트 방법

### 아키텍처

이 예제는 안전한 대화를 위한 가드레일과 메모리의 통합을 보여줍니다:

<div style="text-align:left">
    <img src="guardrails_memory_flow.png" width="90%"/>
</div>

## 0. 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
* Python 3.10+
* AgentCore 메모리 및 Amazon Bedrock에 접근할 수 있는 AWS 자격 증명 설정
* Amazon Bedrock 모델 접근 권한 (Claude Haiku 4.5)
* Amazon Bedrock Memory SDK

먼저 필요한 라이브러리를 설치합니다:

In [ ]:
!pip install -qr requirements.txt

In [ ]:
# 임포트
import os
import boto3
import uuid
import logging
from typing import Dict
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from botocore.exceptions import ClientError
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from strands.experimental.hooks import AfterModelInvocationEvent
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig

# 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("secure-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2') # 에이전트의 AWS 리전
bedrock_client = boto3.client('bedrock', region_name=REGION)
bedrock_runtime_client = boto3.client('bedrock-runtime', region_name=REGION)
memory_client = MemoryClient(region_name=REGION)

## 1. Amazon Bedrock 가드레일 생성

이 섹션에서는 에이전트의 콘텐츠 안전 정책을 적용하기 위한 가드레일을 생성합니다. 가드레일은 사용자 입력과 모델 출력 모두에 적용할 수 있는 안전 필터 역할을 합니다. 이 예제에서는 두 가지 특정 정책을 가진 가드레일을 생성합니다:

1. **입력 필터링**: 사용자의 모욕적인 언어 차단
2. **출력 필터링**: 모델이 정치 관련 주제를 논의하는 것 방지

이 접근 방식은 대화의 양방향에서 다양한 유형의 문제 콘텐츠를 가드레일이 어떻게 보호할 수 있는지 보여줍니다. 입력 필터는 존중하는 대화 환경을 유지하는 데 도움이 되며, 출력 필터는 모델이 잠재적으로 민감한 주제를 논의하지 않도록 합니다.

최종 목표는 메모리에 원치 않는 메시지가 저장되는 것을 방지하여 적절한 콘텐츠만 향후 컨텍스트로 저장되도록 하는 것입니다.

In [ ]:
# 이 요청의 고유 식별자
unique_id = str(uuid.uuid4())[:6]

# 가드레일 설정 정의
guardrail_name = f"SecureConversationGuardrail_{unique_id}"
guardrail_description = "Blocks insults in input and political content in output"

try:
    # 가드레일 생성
    response = bedrock_client.create_guardrail(
        name=guardrail_name,
        description=guardrail_description,
        # 입력에서 모욕적 언어 차단
        contentPolicyConfig={
            'filtersConfig': [
                {
                    'type': 'INSULTS',
                    'inputStrength': 'MEDIUM',
                    'outputStrength': 'MEDIUM',
                    'inputModalities': ['TEXT'],
                    'outputModalities': ['TEXT'],
                    'inputAction': 'BLOCK',
                    'outputAction': 'NONE',
                    'inputEnabled': True,
                    'outputEnabled': False
                }
            ],
            'tierConfig': {
                'tierName': 'CLASSIC'
            }
        },
        # 출력에서 정치 관련 콘텐츠 차단
        topicPolicyConfig={
            'topicsConfig': [
                {
                    'name': 'Politics',
                    'definition': 'Content related to political leaders, elections, political parties, or government affairs',
                    'examples': [
                        'Who is the current president?',
                        'Tell me about the upcoming election',
                        'Explain the political situation in Congress'
                    ],
                    'type': 'DENY',
                    'inputAction': 'NONE',
                    'outputAction': 'BLOCK',
                    'inputEnabled': False,
                    'outputEnabled': True
                }
            ],
            'tierConfig': {
                'tierName': 'CLASSIC'
            }
        },
        blockedInputMessaging="I'm sorry, but your message contains inappropriate language. Please rephrase your question without insults.",
        blockedOutputsMessaging="I apologize, but I cannot provide information on political topics. Is there something else I can help you with?",
    )
    
    # 나중에 사용할 가드레일 ID 저장
    guardrail_id = response['guardrailId']
    guardrail_arn = response['guardrailArn']
    guardrail_version = "DRAFT"  # 새 가드레일은 DRAFT로 생성됨
    
    print(f"✅ 가드레일 생성 완료: {guardrail_id} (ARN: {guardrail_arn})")
    
except Exception as e:
    print(f"❌ 가드레일 생성 오류: {e}")
    # 가드레일이 이미 존재하는 경우 ID를 찾아봄
    try:
        response = bedrock_client.list_guardrails()
        existing_guardrail = next((g for g in response['guardrailSummaries'] 
                                  if g['name'] == guardrail_name), None)
        if existing_guardrail:
            guardrail_id = existing_guardrail['guardrailId']
            guardrail_version = "DRAFT"  # DRAFT 버전 사용
            print(f"기존 가드레일 사용: {guardrail_id}")
    except Exception as list_error:
        print(f"❌ 가드레일 목록 조회 오류: {list_error}")
        guardrail_id = None
        guardrail_version = None

## 2. 메모리 리소스 생성

이 섹션에서는 대화 기록을 저장하기 위한 에이전트의 메모리 리소스를 생성합니다. 메모리를 통해 에이전트는 과거 상호작용을 회상하고, 컨텍스트를 유지하며, 시간이 지남에 따라 더 일관된 응답을 제공할 수 있습니다. 메모리와 가드레일을 결합하면 적절한 콘텐츠만 향후 참조를 위해 저장되도록 할 수 있습니다.

이 예제에서는 추가 전략 없이 간단한 단기 메모리 리소스를 생성합니다. 이는 세션 내에서 대화 컨텍스트를 유지하는 데 적합합니다. 메모리는 가드레일 검사를 통과한 메시지만 저장하여 부적절한 콘텐츠가 필터링되도록 합니다.

In [ ]:
memory_name = f"SecureAgentMemory_{unique_id}"

try:
    # 전략 없이 메모리 리소스 생성 (단기 메모리만 사용)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리를 위해 전략 없음
        description="Short-term memory for personal agent with guardrails",
        event_expiry_days=7,
    )
    memory_id = memory['id']
    logger.info(f"✅ 메모리 생성 완료: {memory_id}")
except ClientError as e:
    logger.info(f"❌ 오류: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하는 경우 ID를 가져옴
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 표시
    logger.error(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if 'memory_id' in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")

## 3. Bedrock 가드레일, Strands 및 AgentCore 메모리 통합

이 섹션에서는 가드레일과 메모리 기능을 통합하는 커스텀 훅을 생성합니다. 구현 내용은 다음과 같습니다:

1. Amazon Bedrock 가드레일을 사용하여 사용자 입력과 모델 출력 모두 검사
2. 부적절한 콘텐츠를 안전한 대체 메시지로 교체
3. 가드레일 검사를 통과한 메시지만 메모리에 저장
4. 에이전트 초기화 시 메모리에서 과거 대화 컨텍스트 검색

이 접근 방식을 통해 에이전트가 메모리 기능의 이점을 누리면서도 깨끗한 대화 기록을 유지할 수 있습니다. 필요한 구성 요소를 구축해 보겠습니다:

In [ ]:
class GuardrailsEvaluator:
    """재사용 가능한 가드레일 평가 유틸리티."""
    
    def __init__(self, guardrail_id: str, guardrail_version: str):
        """가드레일 평가기를 초기화합니다.
        
        Args:
            guardrail_id: 사용할 가드레일의 ID
            guardrail_version: 가드레일 버전 (예: "DRAFT")
        """
        self.guardrail_id = guardrail_id
        self.guardrail_version = guardrail_version
    
    def evaluate_content(self, content: str, source: str) -> Dict:
        """Bedrock 가드레일을 사용하여 콘텐츠를 평가하고 결과를 반환합니다.
        
        Args:
            content: 평가할 텍스트 콘텐츠
            source: 소스 유형 ("INPUT" 또는 "OUTPUT")
            
        Returns:
            가드레일 평가 결과를 포함하는 Dict
        """
        try:
            logger.info(f"⏳ {source} 검사 중: '{content[:30]}...'")
            
            response = bedrock_runtime_client.apply_guardrail(
                guardrailIdentifier=self.guardrail_id,
                guardrailVersion=self.guardrail_version,
                source=source,
                content=[{"text": {"text": content}}]
            )
            
            action = response.get('action')
            logger.info(f"🔍 가드레일 조치: {action}")
            
            return response
        except Exception as e:
            logger.error(f"❌ 가드레일 평가 실패: {e}")
            return {"error": str(e)}


class GuardrailsHookProvider(HookProvider):
    """가드레일 적용과 메모리 저장을 결합하는 훅 프로바이더."""
    
    def __init__(self, guardrails_evaluator: GuardrailsEvaluator):
        self.evaluator = guardrails_evaluator
        self.blocked_outputs = set()

    def after_model_invocation(self, event: AfterModelInvocationEvent) -> None:
        """모델 출력을 가드레일로 검사하고 필요시 교체합니다.
        
        Args:
            event: 모델 응답을 포함하는 이벤트
        """
        # 모델 호출이 실패한 경우 건너뜀
        if event.exception is not None or event.stop_response is None:
            logger.error("⚠️ 모델 호출 실패, 가드레일 검사 건너뜀")
            return
        
        logger.info("🔍 AfterModelInvocationEvent: 모델 출력 검사 중")
        
        # 모델 응답에서 메시지 추출
        message = event.stop_response.message
        
        # 콘텐츠 추출
        if isinstance(message.get("content"), list):
            content = "".join(block.get("text", "") for block in message.get("content", []))
        else:
            content = str(message.get("content", ""))
        
        content_id = hash(content)
        
        # 가드레일로 검사
        result = self.evaluator.evaluate_content(content, "OUTPUT")
        
        # 가드레일 위반 처리
        if result.get("action") == "GUARDRAIL_INTERVENED":
            logger.warning("⛔ 어시스턴트 메시지가 가드레일에 의해 차단됨")
            
            # 이 출력을 차단됨으로 표시
            self.blocked_outputs.add(content_id)
            
            # 가드레일이 제공한 대체 메시지 가져오기 (가능한 경우)
            replacement_content = None
            if "outputs" in result and result["outputs"] and len(result["outputs"]) > 0:
                if "text" in result["outputs"][0]:
                    replacement_content = result["outputs"][0]["text"]
            
            # 대체 메시지가 없는 경우 기본 메시지로 대체
            if not replacement_content:
                replacement_content = "I apologize, but I cannot provide the requested information as it would violate our content policies."
            
            # 메시지 콘텐츠 업데이트 - 사용자에게 보이는 내용이 변경됨
            if isinstance(message.get("content"), list):
                message["content"] = [{"text": replacement_content}]
            else:
                message["content"] = replacement_content
            
            logger.info(f"⚠️ 어시스턴트 메시지를 가드레일 응답으로 교체: {replacement_content[:30]}...")
    
    
    def register_hooks(self, registry: HookRegistry):
        """레지스트리에 모든 훅을 등록합니다.
        
        Args:
            registry: 등록할 훅 레지스트리
        """
        registry.add_callback(AfterModelInvocationEvent, self.after_model_invocation)

## 4. 에이전트 생성 및 설정

이 섹션에서는 지금까지 구축한 모든 구성 요소(Bedrock 모델, 가드레일 평가기, 메모리 지원 훅 프로바이더)를 결합하여 안전한 대화형 에이전트를 생성합니다. 이 통합을 통해 콘텐츠 정책을 적용하고 적절한 컨텍스트를 저장하면서 대화를 유지할 수 있는 완전한 에이전트가 만들어집니다.

In [ ]:
ACTOR_ID = "user_1"
SESSION_ID = "session_001"
#bedrock_model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")

evaluator = GuardrailsEvaluator(
    guardrail_id=guardrail_id,
    guardrail_version=guardrail_version
)

session_manager = None

def create_personal_agent():
    """메모리와 가드레일이 적용된 개인 에이전트 생성"""
    global session_manager
     # 이전 세션 매니저가 존재하면 종료
    if session_manager is not None:
        session_manager.close()

    # AgentCore 메모리 설정
    config = AgentCoreMemoryConfig(
        memory_id=memory_id,
        session_id=SESSION_ID,
        actor_id=ACTOR_ID
    )

    # 세션 매니저 생성 (명시적 라이프사이클 — 정리 셀에서 종료)
    session_manager = AgentCoreMemorySessionManager(
        agentcore_memory_config=config,
        region_name=REGION
    )

    # 세션 매니저와 가드레일 훅으로 에이전트 생성
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
        system_prompt="You are a helpful personal assistant. Be friendly and professional.",
        session_manager=session_manager,
        hooks=[GuardrailsHookProvider(evaluator)],
        callback_handler=None
    )
    return agent
# 에이전트 생성
agent = create_personal_agent()
logger.info("✅ 메모리와 가드레일이 적용된 개인 에이전트 생성 완료")

이 구현은 다음과 같은 기능을 가진 안전한 에이전트를 생성합니다:

1. 초기화 시 메모리에서 기존 대화 컨텍스트를 로드
2. 처리 전에 가드레일로 사용자 입력 검사
3. 사용자에게 보여주기 전에 가드레일로 모델 출력 검사
4. 승인된 메시지만 메모리에 저장하여 향후 컨텍스트로 활용
5. 여러 상호작용에 걸쳐 대화 기록 유지

가드레일과 메모리의 결합을 통해 에이전트가 안전하면서도 컨텍스트를 유지하는 대화 경험을 제공합니다.

## 5. 보안 에이전트 테스트

다양한 유형의 입력으로 에이전트를 테스트하여 가드레일과 메모리 통합이 실제로 어떻게 작동하는지 확인해 보겠습니다. 허용되는 입력과 가드레일 개입을 유발할 수 있는 입력을 모두 시도하여 구현이 올바르게 작동하는지 검증합니다.

먼저 가드레일 검사 및 에이전트 호출을 처리하는 헬퍼 함수를 생성합니다:

In [ ]:
def process_with_guardrails(user_input):
    """에이전트에 전송하기 전에 가드레일로 사용자 입력을 처리합니다.
    
    Args:
        user_input: 사용자의 텍스트 입력
        
    Returns:
        에이전트 응답 또는 가드레일 거부 메시지
    """
    # 가드레일로 입력 검사
    result = evaluator.evaluate_content(user_input, "INPUT")
    
    if result.get("action") == "GUARDRAIL_INTERVENED":
        # 가드레일에서 거부 메시지 가져오기
        if "outputs" in result and result["outputs"] and "text" in result["outputs"][0]:
            rejection_content = result["outputs"][0]["text"]
        else:
            rejection_content = "I cannot process that request."
        
        # 에이전트 호출 없이 거부 메시지 반환
        print(rejection_content)
        return rejection_content
    else:
        # 입력이 가드레일을 통과하면 에이전트 호출 진행
        response = agent(user_input)
        print(response)
        return response

### 테스트 1: 일반 대화

모든 가드레일을 통과하는 일반적인 인사로 시작해 보겠습니다:

In [ ]:
print("테스트 1: 일반 인사")
user_input = "I am dani."
process_with_guardrails(user_input)

### 테스트 2: 모욕적 콘텐츠 (입력 가드레일 트리거 예상)

입력 가드레일에 의해 차단되어야 하는 모욕적 언어가 포함된 입력을 시도해 보겠습니다:

In [ ]:
print("\n테스트 2: 모욕적 콘텐츠 (입력 가드레일이 트리거되어야 함)")
user_input = "You're a stupid assistant."
process_with_guardrails(user_input)

### 테스트 3: 정치 관련 콘텐츠 (출력 가드레일 트리거 예상)

이번에는 정치에 대한 질문을 시도합니다. 이는 입력 가드레일은 통과하지만 출력 가드레일을 트리거해야 합니다:

In [ ]:
print("\n테스트 3: 정치 관련 질문 (출력 가드레일이 트리거되어야 함)")
user_input = "Who is the president of the US?"
process_with_guardrails(user_input)

### 메모리 내용 확인

테스트 후 메모리에 무엇이 저장되었는지 확인해 보겠습니다:

In [ ]:
# 메모리에 저장된 내용 확인
print("\n=== 메모리 내용 ===")
manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
session = manager.create_memory_session(actor_id=ACTOR_ID, session_id=SESSION_ID)
recent_turns = session.get_last_k_turns(k=5)


for i, turn in enumerate(recent_turns):
    print(f"\n턴 {i+1}:")
    for msg in turn:
        role = msg['role']
        content = msg['content']['text']
        print(f"- {role}: {content[:100]}...")

### 테스트 4: 메모리 테스트를 위한 후속 질문
이전 컨텍스트를 에이전트가 기억하는지 확인하기 위해 후속 질문을 해보겠습니다:

In [ ]:
agent = create_personal_agent()
print("\n테스트 4: 메모리 테스트를 위한 후속 질문")
user_input = "What's my name?"
process_with_guardrails(user_input)

## 6. 정리 (선택 사항)

보안 에이전트 실험을 마친 후 이 튜토리얼에서 생성한 리소스를 정리할 수 있습니다. 이 섹션에서는 가드레일과 메모리 리소스를 삭제하는 방법을 보여줍니다.

In [ ]:
# 버퍼링된 메시지를 플러시하기 위해 세션 매니저 종료
if session_manager is not None:
    session_manager.close()
    print("✅ 세션 매니저 종료 완료")

# 메모리 리소스 삭제
try:
    memory_client.delete_memory_and_wait(memory_id=memory_id)
    print(f"✅ 메모리 리소스 삭제 완료: {memory_id}")
except Exception as e:
    print(f"❌ 메모리 삭제 오류: {e}")

# 가드레일 삭제
try:
    bedrock_client.delete_guardrail(
        guardrailIdentifier=guardrail_id
    )
    print(f"✅ 가드레일 삭제 완료: {guardrail_id}")
except Exception as e:
    print(f"❌ 가드레일 삭제 오류: {e}")

## 결론

이 튜토리얼에서는 Amazon Bedrock 가드레일과 AgentCore 메모리 기능을 결합한 안전한 대화형 에이전트를 구축했습니다. 구현 내용:

1. 가드레일을 사용하여 부적절한 사용자 입력 필터링
2. 에이전트가 민감한 주제를 논의하는 것 방지
3. 승인된 메시지만 메모리에 저장
4. 향상된 사용자 경험을 위해 메모리를 활용한 대화 컨텍스트 유지

가드레일과 메모리를 통합함으로써 콘텐츠 정책을 준수하면서도 개인화되고 컨텍스트에 맞는 응답을 제공하는 강력한 에이전트를 구축할 수 있습니다. 이 패턴은 추가적인 가드레일 필터를 추가하거나 더 발전된 장기 메모리 전략을 구현하여 더 복잡한 시나리오로 확장할 수 있습니다.